In [1]:
import pandas as pd

In [ ]:
df1 = pd.read_csv("/content/건강_병원_전국.csv", encoding = "cp949")

In [ ]:
df1.columns

In [ ]:
df2 = pd.read_csv("/content/국토교통부_전국 버스정류장 위치정보_20251031.csv", encoding="cp949")

In [ ]:
df2.columns

In [ ]:
df3 = pd.read_excel("/content/센서스 공간정보 지역 코드 행정동.xlsx")

In [ ]:
df3.columns

In [ ]:
df3.head()

In [ ]:
df4 = pd.read_csv("/content/행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260930.csv", encoding = "cp949")

In [ ]:
df4.columns

# 확인

In [ ]:
import re

In [ ]:
hosp  = df1
stops = df2
line = df3
pop   = df4

T = ['공주시', '고창군', '김천시']
pat = '공주시|고창군|김천시'

In [ ]:
line.columns = line.iloc[0]
line = line.iloc[1:].reset_index(drop=True)
line['시도코드']   = line['시도코드'].astype(str).str.zfill(2)
line['시군구코드'] = line['시군구코드'].astype(str).str.zfill(3)
line['읍면동코드'] = line['읍면동코드'].astype(str).str.zfill(3)
line['adm_cd8'] = line['시도코드'] + line['시군구코드'] + line['읍면동코드']

print(line[line['시군구명칭'].isin(T)].groupby('시군구명칭').size())

# 김천시는 22

In [ ]:
h = hosp[hosp['도로명주소'].fillna('').str.contains(pat) | hosp['지번주소'].fillna('').str.contains(pat)]
print(len(h))
for c in ['영업상태명', '의료기관종별명', '업태구분명']:
    print(h[c].value_counts(dropna=False), '\n')
print(h[['좌표정보(X)', '좌표정보(Y)']].isna().sum())

In [ ]:
s = stops[stops['도시명'].isin(T)]
print(s.groupby(['도시명', '도시코드']).size())

print(stops['도시명'].astype(str).str.contains('공주|고창|김천').sum())
print(stops.loc[stops['도시명'].astype(str).str.contains('공주|고창|김천'), '도시명'].unique())

In [ ]:
cols65 = [c for c in pop.columns if re.match(r'^\d+세', c) and int(re.match(r'^(\d+)세', c).group(1)) >= 65]
print(len(cols65), cols65[:2], cols65[-2:])
print(pop[pop['시군구명'].isin(T)].groupby('시군구명').size())

In [ ]:
T = ['공주시', '고창군', '김천시']

In [ ]:
# 1) 병원: 시군별·업태별·영업상태 교차표
h2 = h.copy()
h2['주소'] = h2['도로명주소'].fillna('') + ' ' + h2['지번주소'].fillna('')
h2['시군구'] = h2['주소'].str.extract(r'(공주시|고창군|김천시)')[0]
print(pd.crosstab([h2['시군구'], h2['업태구분명']], h2['영업상태명']))

tgt = h2[h2['업태구분명'].isin(['병원', '종합병원']) & (h2['영업상태명'] == '영업/정상')]
print(tgt[['시군구','사업장명','업태구분명','병상수','좌표정보(X)','좌표정보(Y)','도로명주소']].to_string())

In [ ]:
# 2) 정류장: 도시명에 도 이름이 붙어 있음
m = stops['도시명'].astype(str).str.contains('공주|고창|김천')
print(stops[m].groupby(['도시명', '도시코드', '관리도시명']).size())
print(stops[m][['위도', '경도']].describe())

In [ ]:
# 3) 읍면동 이름 일치 여부 (코드표 vs 인구)
a = line[line['시군구명칭'].isin(T)][['시군구명칭', '읍면동명칭']]
b = pop[pop['시군구명'].isin(T)][['시군구명', '읍면동명']]
sa, sb = set(zip(a['시군구명칭'], a['읍면동명칭'])), set(zip(b['시군구명'], b['읍면동명']))
print('코드표에만:', sorted(sa - sb))
print('인구에만:', sorted(sb - sa))

In [ ]:
# 4) 코드 체계 비교 (공주시만)
print(pop[pop['시군구명'] == '공주시'][['행정기관코드', '읍면동명']].head(3))
print(line[line['시군구명칭'] == '공주시'][['adm_cd8', '읍면동명칭']].head(3))

# 좌표 수정

In [ ]:
from pyproj import Transformer

In [ ]:
# 1) 병원 목적지 확정 (중복 제거 + 좌표 변환)
tgt2 = tgt.copy()
tgt2['gx'] = tgt2['좌표정보(X)'].round(0)
tgt2['gy'] = tgt2['좌표정보(Y)'].round(0)
tgt2 = tgt2.drop_duplicates(subset=['시군구', 'gx', 'gy']).reset_index(drop=True)
t = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)
tgt2['lon'], tgt2['lat'] = t.transform(tgt2['좌표정보(X)'].values, tgt2['좌표정보(Y)'].values)
print(tgt2[['시군구', '사업장명', '업태구분명', 'lat', 'lon']])

In [ ]:
# 2) 정류장 정리
m = stops['도시명'].astype(str).str.contains('공주|고창|김천')
st = stops[m].copy()
st['시군구'] = st['도시명'].str.extract(r'(공주시|고창군|김천시)')[0]
print(st.groupby('시군구').size())
print('같은 좌표 중복 정류장:', st.duplicated(subset=['위도', '경도']).sum())

In [ ]:
# 3) 읍면동별 65세 이상 인구
p = pop[pop['시군구명'].isin(T)].copy()
for c in cols65 + ['계']:
    p[c] = pd.to_numeric(p[c].astype(str).str.replace(',', ''), errors='coerce')
p['pop65'] = p[cols65].sum(axis=1)
p['ratio65'] = p['pop65'] / p['계']
print(p.groupby('시군구명')[['계', 'pop65']].sum().assign(ratio=lambda d: d['pop65'] / d['계']))
print(p.sort_values('ratio65', ascending=False)[['시군구명', '읍면동명', '계', 'pop65', 'ratio65']].head(10))
print('이름 중복 수(0이어야 이름 조인 가능):', p.duplicated(subset=['시군구명', '읍면동명']).sum())

In [ ]:
import geopandas as gpd, pandas as pd

In [ ]:
# 정류장·병원을 미터 단위 좌표계(EPSG:5179)로 변환
stg = gpd.GeoDataFrame(st, geometry=gpd.points_from_xy(st['경도'], st['위도']), crs='EPSG:4326').to_crs('EPSG:5179')
hg  = gpd.GeoDataFrame(tgt2, geometry=gpd.points_from_xy(tgt2['lon'], tgt2['lat']), crs='EPSG:4326').to_crs('EPSG:5179')

In [ ]:
# 병원별: 최근접 정류장 거리, 반경 내 정류장 수 (같은 시군 정류장만)
rows = []
for _, r in hg.iterrows():
    sub_s = stg[stg['시군구'] == r['시군구']]
    d = sub_s.geometry.distance(r.geometry)
    rows.append({'시군구': r['시군구'], '병원': r['사업장명'],
                 '최근접정류장_m': round(d.min()),
                 '300m내': int((d <= 300).sum()), '500m내': int((d <= 500).sum())})
print(pd.DataFrame(rows).to_string())

# 특수확장자 확인

In [ ]:
import os, geopandas as gpd

In [ ]:
for f in sorted(os.listdir('/content')):
    if 'BND_ADM' in f:
        print(f, round(os.path.getsize('/content/' + f) / 1024), 'KB')

shp = '/content/BND_ADM_DONG_PG.shp'
gdf = gpd.read_file(shp, encoding='cp949')
print(len(gdf), gdf.crs)
print(gdf.columns.tolist())

In [ ]:
# 기준일, 코드 길이, 이름 샘플
print(gdf['BASE_DATE'].unique()[:3])
print(gdf['ADM_CD'].astype(str).str.len().value_counts())
print(gdf[['ADM_CD', 'ADM_NM']].sample(3, random_state=0))

In [ ]:
gdf['adm_cd8'] = gdf['ADM_CD'].astype(str).str[:8]
want = line.loc[line['시군구명칭'].isin(T), 'adm_cd8']
print('코드표 대상 읍면동:', len(want), '| 경계 파일에서 일치:', want.isin(gdf['adm_cd8']).sum())

In [ ]:
codes5 = want.str[:5].unique()
sub = gdf[gdf['adm_cd8'].str[:5].isin(codes5)].copy().to_crs('EPSG:5179')
print(len(sub)); print(sub.groupby(sub['adm_cd8'].str[:5]).size())
print('코드표에만 있는 코드:', set(want) - set(sub['adm_cd8']))
print('경계에만 있는 코드:', set(sub['adm_cd8']) - set(want))

# 병합 및 추가 작업

In [ ]:
for v in ['p', 'stg', 'tgt2']:
    print(v, v in globals())
print('좌표계 확인:', stg.crs, '|', sub.crs)

In [ ]:
# 코드표 이름과 인구 붙이기
sub = sub.merge(line[['adm_cd8', '시군구명칭', '읍면동명칭']], on='adm_cd8', how='left')
p2 = p.rename(columns={'시군구명': '시군구명칭', '읍면동명': '읍면동명칭'})[['시군구명칭', '읍면동명칭', '계', 'pop65', 'ratio65']]
sub = sub.merge(p2, on=['시군구명칭', '읍면동명칭'], how='left')
print('인구 결측:', sub['pop65'].isna().sum())

In [ ]:
# 정류장을 읍면동에 배정
import geopandas as gpd
j = gpd.sjoin(stg, sub[['adm_cd8', 'geometry']], how='left', predicate='within')
print('읍면동에 배정 안 된 정류장:', j['adm_cd8'].isna().sum(), '/', len(j))
cnt = j.groupby('adm_cd8').size().rename('n_stops')
sub = sub.merge(cnt, on='adm_cd8', how='left').fillna({'n_stops': 0})

In [ ]:
# 읍면동 대표점에서 최근접 정류장까지 거리(m)
rp = sub[['adm_cd8', 'geometry']].copy()
rp['geometry'] = rp.geometry.representative_point()
near = gpd.sjoin_nearest(rp, stg[['geometry']], distance_col='d_near_m')
sub = sub.merge(near.groupby('adm_cd8')['d_near_m'].min().round().rename('d_near_m'), on='adm_cd8', how='left')

print(sub['d_near_m'].describe())
print(sub.sort_values('d_near_m', ascending=False)[['시군구명칭', '읍면동명칭', 'pop65', 'n_stops', 'd_near_m']].head(10).to_string())
print('정류장 0개 읍면동:', sub.loc[sub['n_stops'] == 0, ['시군구명칭', '읍면동명칭']].values.tolist())

In [ ]:
print(type(sub), sub.crs, len(sub))

In [ ]:
# 1) 시군별 배정 안 된 비율
un_flag = j['adm_cd8'].isna()
print(j.assign(un=un_flag).groupby('시군구')['un'].agg(['sum', 'count', 'mean']))

In [ ]:
# 2) 배정 안 된 정류장이 전국 경계 기준으로 어느 시군구에 속하는가
allg = gdf.to_crs('EPSG:5179')
un = gpd.GeoDataFrame(j.loc[un_flag, ['시군구', 'geometry']], geometry='geometry', crs='EPSG:5179')
w = gpd.sjoin(un, allg[['adm_cd8', 'geometry']], how='left', predicate='within')
w['sgg5'] = w['adm_cd8'].astype(str).str[:5]

line['sgg5'] = line['시도코드'] + line['시군구코드']
nm = line.drop_duplicates('sgg5').set_index('sgg5')['시군구명칭']
w['속한시군구'] = w['sgg5'].map(nm).fillna('경계 밖')
print(pd.crosstab(w['시군구'], w['속한시군구']).T.sort_values('공주시', ascending=False).head(15))

In [ ]:
# 3) 경계 밖이면 대상 읍면동에서 얼마나 떨어져 있나
d_out = gpd.sjoin_nearest(un, sub[['adm_cd8', 'geometry']], distance_col='d_out')
print(d_out['d_out'].describe())
print((d_out['d_out'] <= 100).sum(), '개가 경계에서 100m 이내')

In [ ]:
pop['기준연월'].unique()

In [ ]:
hg = gpd.GeoDataFrame(tgt2, geometry=gpd.points_from_xy(tgt2['lon'], tgt2['lat']), crs='EPSG:4326').to_crs('EPSG:5179')
rp = sub[['adm_cd8', '시군구명칭', 'geometry']].copy()
rp['geometry'] = rp.geometry.representative_point()

In [ ]:
rows = []
for _, r in rp.iterrows():
    cand = hg[hg['시군구'] == r['시군구명칭']]
    d = cand.geometry.distance(r.geometry)
    rows.append({'adm_cd8': r['adm_cd8'], 'd_hosp_m': round(d.min()), '최근접병원': cand.loc[d.idxmin(), '사업장명']})
sub = sub.merge(pd.DataFrame(rows), on='adm_cd8', how='left')
print(sub['d_hosp_m'].describe())
print(sub.sort_values('d_hosp_m', ascending=False)[['시군구명칭', '읍면동명칭', 'pop65', 'd_hosp_m', 'd_near_m', 'cover500']].head(10).to_string())

In [ ]:
print(stops.loc[stops['도시명'].astype(str).str.contains('공주|고창|김천'), '정보수집일'].agg(['min', 'max']))
print(hosp['데이터갱신시점'].max(), '|', hosp['최종수정시점'].max())

# 이미지1

In [ ]:
import itertools

In [ ]:
# 1) 시군 밖 병원 포함: 전국 병원급 영업 중 기준 최근접 거리
H = hosp[hosp['업태구분명'].isin(['병원', '종합병원']) & (hosp['영업상태명'] == '영업/정상')]
H = H.dropna(subset=['좌표정보(X)', '좌표정보(Y)'])
H = H[(H['좌표정보(X)'] > 0) & (H['좌표정보(Y)'] > 0)]
print('전국 병원급(영업):', len(H))
hg_all = gpd.GeoDataFrame(H, geometry=gpd.points_from_xy(H['좌표정보(X)'], H['좌표정보(Y)']), crs='EPSG:5174').to_crs('EPSG:5179')

rp = sub[['adm_cd8', 'geometry']].copy()
rp['geometry'] = rp.geometry.representative_point()
nh = gpd.sjoin_nearest(rp, hg_all[['사업장명', 'geometry']], distance_col='d_hosp_any_m')
nh = nh.sort_values('d_hosp_any_m').drop_duplicates('adm_cd8')[['adm_cd8', 'd_hosp_any_m', '사업장명']].rename(columns={'사업장명': '최근접병원_전국'})
sub = sub.merge(nh, on='adm_cd8', how='left')
sub['차이_km'] = ((sub['d_hosp_m'] - sub['d_hosp_any_m']) / 1000).round(1)
print(sub.sort_values('차이_km', ascending=False)[['시군구명칭', '읍면동명칭', 'd_hosp_m', 'd_hosp_any_m', '최근접병원_전국', '차이_km']].head(10).to_string())

In [ ]:
# 2) 기존 지표끼리 비교(기준선)
cols = ['d_near_m', 'cover500', 'd_hosp_m', 'd_hosp_any_m', 'ratio65', 'pop65']
print(sub[cols].corr(method='spearman').round(2))
tops = {'정류장거리': set(sub.nlargest(10, 'd_near_m')['adm_cd8']),
        '커버리지': set(sub.nsmallest(10, 'cover500')['adm_cd8']),
        '병원거리': set(sub.nlargest(10, 'd_hosp_m')['adm_cd8'])}
for a, b in itertools.combinations(tops, 2):
    print(a, '∩', b, '=', len(tops[a] & tops[b]), '/ 10')

In [ ]:
# 3) 병원 300m 정류장군 저장
hs = []
for _, r in hg.iterrows():
    cand = stg[stg['시군구'] == r['시군구']]
    m = cand[cand.geometry.distance(r.geometry) <= 300].copy()
    m['병원'] = r['사업장명']
    hs.append(m)
hs = pd.concat(hs)
print(hs.groupby('병원').size())
hs[['정류장번호', '정류장명', '위도', '경도', '병원']].to_csv('hospital_stops_300m.csv', index=False, encoding='utf-8-sig')

In [ ]:
# 4) 중간 결과 저장
sub.to_file('stage1.gpkg', driver='GPKG')

In [ ]:
!apt-get -qq -y install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'; plt.rcParams['axes.unicode_minus'] = False

In [ ]:
sub = gpd.read_file('stage1.gpkg')
T = ['공주시', '고창군', '김천시']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, t in zip(axes, T):
    s = sub[sub['시군구명칭'] == t]
    s.plot(column='d_hosp_m', cmap='OrRd', legend=True, edgecolor='gray', linewidth=0.4, ax=ax)
    ax.set_title(f'{t}: 병원까지 직선거리(m)'); ax.axis('off')
plt.tight_layout(); plt.savefig('map_draft.png', dpi=150)

In [ ]:
## 상단 지도 수정본
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'

vmax = sub['d_hosp_m'].max() / 1000
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, t in zip(axes, T):
    s = sub[sub['시군구명칭'] == t].copy(); s['km'] = s['d_hosp_m'] / 1000
    s.plot(column='km', cmap='OrRd', vmin=0, vmax=vmax, edgecolor='gray', linewidth=0.4, ax=ax)
    hg[hg['시군구'] == t].plot(ax=ax, color='navy', marker='*', markersize=140, zorder=5)
    for _, r in s.nlargest(4, 'km').iterrows():
        c = r.geometry.representative_point()
        ax.annotate(r['읍면동명칭'], (c.x, c.y), ha='center', fontsize=9)
    ax.set_title(t); ax.axis('off')
sm = plt.cm.ScalarMappable(cmap='OrRd', norm=plt.Normalize(0, vmax)); sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=0.7, label='가장 가까운 병원까지 직선거리(km)')
fig.text(0.01, 0.01, '★ 병원(영업 중 병원·종합병원, 같은 시군). 경계 2025-06-30, 병원 인허가 2026-10-12', fontsize=8)
plt.savefig('map1_hospital_distance.png', dpi=200, bbox_inches='tight')

- 비교군

In [2]:
import numpy as np

In [ ]:
# (a) 상위 k 겹침 vs 우연 기대치
n = len(sub)
for k in [5, 10, 15, 20]:
    tops = {'정류장거리': set(sub.nlargest(k, 'd_near_m')['adm_cd8']),
            '병원거리': set(sub.nlargest(k, 'd_hosp_m')['adm_cd8']),
            '커버리지': set(sub.nsmallest(k, 'cover500')['adm_cd8'])}
    out = {f'{a}∩{b}': len(tops[a] & tops[b]) for a, b in itertools.combinations(tops, 2)}
    print(k, out, '| 우연 기대치:', round(k * k / n, 1))

In [ ]:
# (b) 시군 밖 병원이 더 가까운 읍면
print('경계 밖 병원이 1km 이상 더 가까운 읍면:', (sub['차이_km'] >= 1).sum(), '/', n)

In [ ]:
# (c) 전국 병원 후보의 진료과목 구성(전문병원 점검)
print(H['진료과목내용명'].astype(str).str.contains('내과').mean())
print(H[H['사업장명'].str.contains('재활|정신|요양|한방|치과', na=False)]['업태구분명'].value_counts())

In [ ]:
##인구산점도

sub['hosp_km'] = sub['d_hosp_m'] / 1000
xm, ym = sub['d_near_m'].median(), sub['hosp_km'].median()
print('중앙값: 정류장거리', xm, 'm | 병원거리', round(ym, 1), 'km')

q = sub[(sub['d_near_m'] <= xm) & (sub['hosp_km'] >= ym)].sort_values('hosp_km', ascending=False)
print('정류장은 가까운데 병원은 먼 읍면:', len(q))
print(q[['시군구명칭', '읍면동명칭', 'pop65', 'd_near_m', 'hosp_km']].to_string())

colors = {'공주시': 'tab:blue', '고창군': 'tab:green', '김천시': 'tab:red'}
fig, ax = plt.subplots(figsize=(9, 7))
for t, g in sub.groupby('시군구명칭'):
    ax.scatter(g['d_near_m'], g['hosp_km'], s=g['pop65'] / 8, alpha=0.6, color=colors[t],
               label=t, edgecolor='k', linewidth=0.3)
ax.axvline(xm, ls='--', c='gray'); ax.axhline(ym, ls='--', c='gray')
for _, r in q.head(8).iterrows():
    ax.annotate(r['읍면동명칭'], (r['d_near_m'], r['hosp_km']), fontsize=9, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)')
ax.set_ylabel('가장 가까운 병원까지 직선거리(km)')
ax.legend(title='시군(점 크기=65세 이상 인구)')
plt.savefig('fig2_scatter.png', dpi=200, bbox_inches='tight')